In [0]:
-- Medallion Architecture Setup: Catalog and Schema Structure
-- Bronze: Raw data ingestion layer (as-is from source)
-- Silver: Cleaned and validated data (quality transformations applied)
-- Gold: Business-level aggregates and star schema (analytics-ready)

CREATE CATALOG IF NOT EXISTS Bakehouse_retail;
USE CATALOG Bakehouse_retail;

CREATE SCHEMA IF NOT EXISTS Bakehouse_retail_bronze;  -- Raw data landing zone
USE Bakehouse_retail_bronze;

CREATE SCHEMA IF NOT EXISTS Bakehouse_retail_silver;  -- Cleaned and standardized data
USE Bakehouse_retail_silver;

CREATE SCHEMA IF NOT EXISTS Bakehouse_retail_gold;    -- Star schema for analytics
USE Bakehouse_retail_gold;

In [0]:
SELECT *
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/customersjson.json',
  format => 'json',
  multiLine => true
)
LIMIT 100

In [0]:
-- Data cleaning: standardize product names with proper capitalization
SELECT 
  product_name AS original_name,
  -- Multi-step transformation: 1) TRIM whitespace, 2) collapse extra spaces, 3) title case, 4) fix acronyms
  REPLACE(
    INITCAP(
      TRIM(
        REGEXP_REPLACE(product_name, '\\s+', ' ')  -- Replace multiple spaces with single space
      )
    ),
    'Usb-c', 'USB-C'  -- Preserve uppercase acronyms after title case
  ) AS cleaned_name,
  COUNT(*) as occurrence_count
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/transactions_*.csv',
  format => 'csv',
  multiLine => true
)
GROUP BY product_name
ORDER BY cleaned_name, original_name

In [0]:
SELECT *
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/productsjson.json',
  format => 'json',
  multiLine => true
)
LIMIT 100

In [0]:
SELECT *
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/transactions_2025_01_06.csv',
  format => 'csv',
  multiLine => true
)
LIMIT 100

In [0]:
SELECT *
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/transactions_2025_01_13.csv',
  format => 'csv',
  multiLine => true
)
LIMIT 10

In [0]:
SELECT *
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/transactions_2025_01_20.csv',
  format => 'csv',
  multiLine => true
)
LIMIT 10

In [0]:
SELECT *
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/transactions_2025_01_27.csv',
  format => 'csv',
  multiLine => true
)
LIMIT 10

In [0]:
-- Data cleaning: standardize product names with proper capitalization
SELECT 
  product_name AS original_name,
  REPLACE(
    INITCAP(
      TRIM(
        REGEXP_REPLACE(product_name, '\\s+', ' ')
      )
    ),
    'Usb-c', 'USB-C'
  ) AS cleaned_name,
  COUNT(*) as occurrence_count
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/transactions_*.csv',
  format => 'csv',
  multiLine => true
)
GROUP BY product_name
ORDER BY cleaned_name, original_name

In [0]:
-- View the 15 unique standardized product names
SELECT DISTINCT
  REPLACE(
    INITCAP(
      TRIM(
        REGEXP_REPLACE(product_name, '\\s+', ' ')
      )
    ),
    'Usb-c', 'USB-C'
  ) AS standardized_product_name,
  SUM(COUNT(*)) OVER (PARTITION BY REPLACE(INITCAP(TRIM(REGEXP_REPLACE(product_name, '\\s+', ' '))), 'Usb-c', 'USB-C')) as total_transactions
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/transactions_*.csv',
  format => 'csv',
  multiLine => true
)
GROUP BY product_name
ORDER BY standardized_product_name

In [0]:
-- Query all distinct product names across all transaction files
SELECT DISTINCT 
  product_name,
  COUNT(*) as occurrence_count
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/transactions_*.csv',
  format => 'csv',
  multiLine => true
)
GROUP BY product_name
ORDER BY product_name

In [0]:
-- Comprehensive data cleaning: trim all string columns and standardize product names
SELECT 
  TRIM(transaction_id) AS transaction_id,
  transaction_date,
  TRIM(customer_id) AS customer_id,
  TRIM(product_id) AS product_id,
  REPLACE(
    INITCAP(
      TRIM(
        REGEXP_REPLACE(product_name, '\\s+', ' ')
      )
    ),
    'Usb-c', 'USB-C'
  ) AS product_name,
  TRIM(category) AS category,
  quantity,
  unit_price,
  total_amount,
  TRIM(store_location) AS store_location,
  TRIM(payment_method) AS payment_method
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/transactions_*.csv',
  format => 'csv',
  multiLine => true
)
LIMIT 20

In [0]:
-- Silver Layer: Cleaned Products Table
CREATE OR REPLACE TABLE bakehouse_retail.bakehouse_retail_silver.products_clean AS
SELECT 
  product_id,
  TRIM(product_name) AS product_name,
  TRIM(category) AS category,
  price,
  CURRENT_TIMESTAMP() AS processed_at
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/productsjson.json',
  format => 'json',
  multiLine => true
);

In [0]:
-- Silver Layer: Cleaned Orders Table
CREATE OR REPLACE TABLE bakehouse_retail.bakehouse_retail_silver.orders_clean AS
SELECT 
  order_id,
  customer_id,
  TRIM(order_date) AS order_date,
  product_id,
  quantity,
  total_amount,
  CURRENT_TIMESTAMP() AS processed_at
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/ordersjson.json',
  format => 'json',
  multiLine => true
);

In [0]:
-- Silver Layer: Cleaned Customers Table
CREATE OR REPLACE TABLE bakehouse_retail.bakehouse_retail_silver.customers_clean AS
SELECT 
  customer_id,
  TRIM(first_name) AS first_name,
  TRIM(last_name) AS last_name,
  TRIM(country) AS country,
  TRIM(signup_date) AS signup_date,
  CURRENT_TIMESTAMP() AS processed_at
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/customersjson.json',
  format => 'json',
  multiLine => true
);

In [0]:
-- Silver Layer: Cleaned Transactions Table
-- Apply comprehensive data quality transformations: trim whitespace, standardize product names
CREATE OR REPLACE TABLE bakehouse_retail.bakehouse_retail_silver.transactions_clean AS
SELECT 
  TRIM(transaction_id) AS transaction_id,
  transaction_date,
  TRIM(customer_id) AS customer_id,
  TRIM(product_id) AS product_id,
  -- Standardize product names: remove extra spaces, apply title case, preserve acronyms
  REPLACE(
    INITCAP(
      TRIM(
        REGEXP_REPLACE(product_name, '\\s+', ' ')
      )
    ),
    'Usb-c', 'USB-C'
  ) AS product_name,
  TRIM(category) AS category,
  quantity,
  unit_price,
  total_amount,
  TRIM(store_location) AS store_location,
  TRIM(payment_method) AS payment_method,
  CURRENT_TIMESTAMP() AS processed_at
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/transactions_*.csv',  -- Wildcard to load all weekly files
  format => 'csv',
  multiLine => true
);

In [0]:
-- Silver Layer Summary: View all cleaned tables
SHOW TABLES IN bakehouse_retail.bakehouse_retail_silver;

In [0]:
-- Verify Silver Layer: Query cleaned transactions with standardized product names
SELECT 
  transaction_id,
  transaction_date,
  customer_id,
  product_name,
  category,
  quantity,
  unit_price,
  total_amount,
  store_location,
  payment_method
FROM bakehouse_retail.bakehouse_retail_silver.transactions_clean
ORDER BY transaction_date DESC
LIMIT 20;

In [0]:
-- Data Quality Check: Null Analysis for Customers Table
-- Pattern: COUNT(*) - COUNT(column) = number of nulls (COUNT ignores nulls)
-- UNION ALL combines null counts for all columns into single report
SELECT 
  'customer_id' as column_name,
  COUNT(*) - COUNT(customer_id) as null_count,
  COUNT(*) as total_rows
FROM bakehouse_retail.bakehouse_retail_silver.customers_clean
UNION ALL
SELECT 
  'first_name' as column_name,
  COUNT(*) - COUNT(first_name) as null_count,
  COUNT(*) as total_rows
FROM bakehouse_retail.bakehouse_retail_silver.customers_clean
UNION ALL
SELECT 
  'last_name' as column_name,
  COUNT(*) - COUNT(last_name) as null_count,
  COUNT(*) as total_rows
FROM bakehouse_retail.bakehouse_retail_silver.customers_clean
UNION ALL
SELECT 
  'country' as column_name,
  COUNT(*) - COUNT(country) as null_count,
  COUNT(*) as total_rows
FROM bakehouse_retail.bakehouse_retail_silver.customers_clean
UNION ALL
SELECT 
  'signup_date' as column_name,
  COUNT(*) - COUNT(signup_date) as null_count,
  COUNT(*) as total_rows
FROM bakehouse_retail.bakehouse_retail_silver.customers_clean
ORDER BY null_count DESC;

In [0]:
-- Check for nulls in orders_clean and products_clean
SELECT 'Orders' as table_name, 'order_id' as column_name, COUNT(*) - COUNT(order_id) as null_count FROM bakehouse_retail.bakehouse_retail_silver.orders_clean
UNION ALL SELECT 'Orders', 'customer_id', COUNT(*) - COUNT(customer_id) FROM bakehouse_retail.bakehouse_retail_silver.orders_clean
UNION ALL SELECT 'Orders', 'product_id', COUNT(*) - COUNT(product_id) FROM bakehouse_retail.bakehouse_retail_silver.orders_clean
UNION ALL SELECT 'Orders', 'order_date', COUNT(*) - COUNT(order_date) FROM bakehouse_retail.bakehouse_retail_silver.orders_clean
UNION ALL SELECT 'Products', 'product_id', COUNT(*) - COUNT(product_id) FROM bakehouse_retail.bakehouse_retail_silver.products_clean
UNION ALL SELECT 'Products', 'product_name', COUNT(*) - COUNT(product_name) FROM bakehouse_retail.bakehouse_retail_silver.products_clean
UNION ALL SELECT 'Products', 'category', COUNT(*) - COUNT(category) FROM bakehouse_retail.bakehouse_retail_silver.products_clean
ORDER BY null_count DESC;

In [0]:
-- Check for nulls in transactions_clean
SELECT 
  'transaction_id' as column_name,
  COUNT(*) - COUNT(transaction_id) as null_count,
  COUNT(*) as total_rows
FROM bakehouse_retail.bakehouse_retail_silver.transactions_clean
UNION ALL
SELECT 
  'customer_id' as column_name,
  COUNT(*) - COUNT(customer_id) as null_count,
  COUNT(*) as total_rows
FROM bakehouse_retail.bakehouse_retail_silver.transactions_clean
UNION ALL
SELECT 
  'product_id' as column_name,
  COUNT(*) - COUNT(product_id) as null_count,
  COUNT(*) as total_rows
FROM bakehouse_retail.bakehouse_retail_silver.transactions_clean
UNION ALL
SELECT 
  'product_name' as column_name,
  COUNT(*) - COUNT(product_name) as null_count,
  COUNT(*) as total_rows
FROM bakehouse_retail.bakehouse_retail_silver.transactions_clean
UNION ALL
SELECT 
  'category' as column_name,
  COUNT(*) - COUNT(category) as null_count,
  COUNT(*) as total_rows
FROM bakehouse_retail.bakehouse_retail_silver.transactions_clean
UNION ALL
SELECT 
  'store_location' as column_name,
  COUNT(*) - COUNT(store_location) as null_count,
  COUNT(*) as total_rows
FROM bakehouse_retail.bakehouse_retail_silver.transactions_clean
UNION ALL
SELECT 
  'payment_method' as column_name,
  COUNT(*) - COUNT(payment_method) as null_count,
  COUNT(*) as total_rows
FROM bakehouse_retail.bakehouse_retail_silver.transactions_clean
ORDER BY null_count DESC;

In [0]:
-- Verify Star Schema: Query fact table with date, location, and payment dimensions
-- Classic star schema join pattern: fact table at center, dimension tables as spokes
-- Note: customers.json (bigint 101-105) and transactions.csv (string C1234 format) have different ID formats
-- Solution: fact table includes denormalized customer_id and product attributes for direct querying
SELECT 
  f.transaction_id,
  -- Date dimension attributes
  d.full_date,
  d.day_name,
  d.month_name,
  d.year,
  d.quarter,
  -- Denormalized attributes (no join needed)
  f.customer_id,
  f.product_name,
  f.category,
  -- Dimension lookups via foreign keys
  l.location_name,
  pm.payment_method_name,
  -- Measures
  f.quantity,
  f.unit_price,
  f.total_amount
FROM bakehouse_retail.bakehouse_retail_gold.fact_sales f
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_date d
  ON f.date_key = d.date_key
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_store_locations l
  ON f.location_key = l.location_key
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_payment_methods pm
  ON f.payment_method_key = pm.payment_method_key
ORDER BY f.transaction_date DESC
LIMIT 10;

Star Schema Gold Layer Summary

Architecture Overview
The Gold layer is implementing a star schema optimized for analytical queries and reporting.

Fact Table (Center of Star)
- fact_sales - Sales transactions with measures (quantity, unit_price, total_amount)
  * Foreign Keys: date_key, location_key, payment_method_key
  * Denormalized: customer_id, product_id, product_name, category (for query performance)

Dimension Tables (Points of Star)
1. dim_date - Time dimension with year, quarter, month, day attributes
2. dim_store_locations - Store location details
3. dim_payment_methods - Payment method types
4. dim_customers - Customer details (separate data source)
5. dim_products - Product catalog (separate data source)

Benefits
* Query Performance - Optimized for OLAP queries with denormalized fact table
* Analytical Flexibility - Easy to slice/dice by time, location, payment method
* Simplified Joins - Star pattern reduces join complexity
* Business Intelligence Ready - Compatible with BI tools like Tableau, Power BI

Data Quality
-  No nulls in any dimension or fact table  
-  Product names standardized (91 variations → 15 clean names)  
-  All string columns trimmed  
- Surrogate keys for location and payment dimensions_

In [0]:
-- Analytical Query Example: Monthly Sales by Category and Location
SELECT 
  d.year,
  d.month_name,
  f.category,
  l.location_name,
  COUNT(*) as transaction_count,
  SUM(f.quantity) as total_quantity,
  ROUND(SUM(f.total_amount), 2) as total_revenue
FROM bakehouse_retail.bakehouse_retail_gold.fact_sales f
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_date d
  ON f.date_key = d.date_key
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_store_locations l
  ON f.location_key = l.location_key
GROUP BY d.year, d.month_name, f.category, l.location_name
ORDER BY d.year, d.month_name, total_revenue DESC
LIMIT 20;

In [0]:
-- Analytical Query Example: Monthly Sales by Category and Location
-- Multi-dimensional analysis: slice by time (month), product (category), and geography (location)
SELECT 
  d.year,
  d.month_name,
  f.category,
  l.location_name,
  -- Aggregated measures
  COUNT(*) as transaction_count,
  SUM(f.quantity) as total_quantity,
  ROUND(SUM(f.total_amount), 2) as total_revenue
FROM bakehouse_retail.bakehouse_retail_gold.fact_sales f
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_date d
  ON f.date_key = d.date_key
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_store_locations l
  ON f.location_key = l.location_key
GROUP BY d.year, d.month_name, f.category, l.location_name
ORDER BY d.year, d.month_name, total_revenue DESC
LIMIT 20;

In [0]:
-- Advanced Analytics: Location-Category Matrix (Top Combinations)
SELECT 
  l.location_name,
  f.category,
  pm.payment_method_name as top_payment_method,
  COUNT(DISTINCT f.transaction_id) as transaction_count,
  SUM(f.quantity) as items_sold,
  ROUND(SUM(f.total_amount), 2) as total_revenue,
  ROUND(AVG(f.total_amount), 2) as avg_transaction_value
FROM bakehouse_retail.bakehouse_retail_gold.fact_sales f
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_store_locations l
  ON f.location_key = l.location_key
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_payment_methods pm
  ON f.payment_method_key = pm.payment_method_key
WHERE f.category IN ('Electronics', 'Apparel', 'Accessories')  -- Top 3 revenue categories
GROUP BY l.location_name, f.category, pm.payment_method_name
HAVING COUNT(DISTINCT f.transaction_id) >= 2  -- At least 2 transactions
ORDER BY total_revenue DESC
LIMIT 15;

Key Business Insights from Star Schema Analytics

Payment Methods
* Cash is King: 27% of revenue ($5,034), highest transaction count (51)
* Digital payments growing: Google Pay + Apple Pay = 29.6% of revenue
* **Opportunity: Promote digital payments to reduce cash handling costs

Top Products
* Running Shoes dominate: $3,600 revenue (19% of total), 45 units sold
* High-value electronics: Mechanical Keyboard ($3,060) and Headphones ($2,040)
* Phone Charger: Most transactions (22) but lower revenue due to $16.99 price
* Margin opportunity: T-shirts have high volume (15 transactions) but low price point

Store Performance
* Orlando leads: $4,291 revenue, highest avg transaction ($110)
* Nashville underperforms: Only $1,841 revenue, lowest avg transaction ($68)
* Consistent basket size: ~3 items per transaction across all locations
* Action item: Investigate why Nashville has 30% lower transaction values

Day of Week Patterns
* Monday is peak: $3,938 revenue (21% of weekly total), highest avg transaction ($119)
* Tuesday slowest: $2,093 revenue, lowest avg transaction ($81)
* Weekend opportunity: Saturday/Sunday combined = 29% of revenue
* Staffing insight: Align staff schedules with Monday/Friday peaks

Category Analysis
* Electronics dominates: 42% of total revenue ($7,852)
* Apparel high-value: 24% of revenue with highest avg transaction ($152)
* Food underperforms: Only 0.5% of revenue despite 14 transactions
* Portfolio balance: Consider expanding high-margin categories (Apparel, Accessories)

Daily Trends
* Peak day: Saturday Jan 25 ($1,518) and Sunday Feb 2 ($1,105)
* Growth pattern: Revenue volatility suggests promotional impact
* Week 2 & 5 stronger: Higher average daily revenue
* Forecasting: Use historical patterns for inventory planning

In [0]:
-- Analytics: Daily Sales Trend
SELECT 
  d.full_date,
  d.day_name,
  d.week_of_year,
  COUNT(DISTINCT f.transaction_id) as transaction_count,
  SUM(f.quantity) as total_items_sold,
  ROUND(SUM(f.total_amount), 2) as daily_revenue,
  ROUND(AVG(f.total_amount), 2) as avg_transaction_value
FROM bakehouse_retail.bakehouse_retail_gold.fact_sales f
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_date d
  ON f.date_key = d.date_key
GROUP BY d.full_date, d.day_name, d.week_of_year
ORDER BY d.full_date;

In [0]:
-- Analytics: Category Performance Summary
-- Product category analysis with revenue share percentage
SELECT 
  f.category,
  COUNT(DISTINCT f.product_name) as unique_products,
  COUNT(DISTINCT f.transaction_id) as transaction_count,
  SUM(f.quantity) as total_items_sold,
  ROUND(SUM(f.total_amount), 2) as total_revenue,
  ROUND(AVG(f.total_amount), 2) as avg_transaction_value,
  ROUND(AVG(f.unit_price), 2) as avg_product_price,
  -- Window function: calculate each category's percentage of total revenue
  ROUND(SUM(f.total_amount) * 100.0 / SUM(SUM(f.total_amount)) OVER (), 2) as revenue_share_pct
FROM bakehouse_retail.bakehouse_retail_gold.fact_sales f
GROUP BY f.category
ORDER BY total_revenue DESC;

In [0]:
-- Analytics: Sales Pattern by Day of Week
-- Identify strongest and weakest days for operational planning (staffing, inventory, promotions)
SELECT 
  d.day_of_week,  -- 1=Sunday, 2=Monday, etc.
  d.day_name,
  COUNT(DISTINCT f.transaction_id) as transaction_count,
  SUM(f.quantity) as total_items_sold,
  ROUND(SUM(f.total_amount), 2) as total_revenue,
  ROUND(AVG(f.total_amount), 2) as avg_transaction_value,
  -- Each day's percentage contribution to total weekly revenue
  ROUND(SUM(f.total_amount) * 100.0 / SUM(SUM(f.total_amount)) OVER (), 2) as revenue_percentage
FROM bakehouse_retail.bakehouse_retail_gold.fact_sales f
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_date d
  ON f.date_key = d.date_key
GROUP BY d.day_of_week, d.day_name
ORDER BY d.day_of_week;

In [0]:
-- Analytics: Store Location Performance Comparison
SELECT 
  l.location_name,
  COUNT(DISTINCT f.transaction_id) as transaction_count,
  COUNT(DISTINCT f.customer_id) as unique_customers,
  SUM(f.quantity) as total_items_sold,
  ROUND(SUM(f.total_amount), 2) as total_revenue,
  ROUND(AVG(f.total_amount), 2) as avg_transaction_value,
  ROUND(SUM(f.quantity) * 1.0 / COUNT(DISTINCT f.transaction_id), 2) as avg_items_per_transaction
FROM bakehouse_retail.bakehouse_retail_gold.fact_sales f
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_store_locations l
  ON f.location_key = l.location_key
GROUP BY l.location_name
ORDER BY total_revenue DESC;

In [0]:
-- Analytics: Top 10 Products by Revenue
-- Product performance ranking with revenue contribution percentage
SELECT 
  f.product_name,
  f.category,
  COUNT(*) as transaction_count,
  SUM(f.quantity) as total_quantity_sold,
  ROUND(AVG(f.unit_price), 2) as avg_unit_price,
  ROUND(SUM(f.total_amount), 2) as total_revenue,
  -- Calculate each product's share of total revenue using window function
  ROUND(SUM(f.total_amount) * 100.0 / SUM(SUM(f.total_amount)) OVER (), 2) as revenue_share_pct
FROM bakehouse_retail.bakehouse_retail_gold.fact_sales f
GROUP BY f.product_name, f.category
ORDER BY total_revenue DESC
LIMIT 10;

In [0]:
-- Analytics: Payment Method Performance Analysis
-- Compare payment method adoption and revenue contribution
SELECT 
  pm.payment_method_name,
  COUNT(DISTINCT f.transaction_id) as transaction_count,
  SUM(f.quantity) as total_items_sold,
  ROUND(SUM(f.total_amount), 2) as total_revenue,
  ROUND(AVG(f.total_amount), 2) as avg_transaction_value,
  -- Revenue share percentage using window function over all payment methods
  ROUND(SUM(f.total_amount) * 100.0 / SUM(SUM(f.total_amount)) OVER (), 2) as revenue_share_pct
FROM bakehouse_retail.bakehouse_retail_gold.fact_sales f
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_payment_methods pm
  ON f.payment_method_key = pm.payment_method_key
GROUP BY pm.payment_method_name
ORDER BY total_revenue DESC;

In [0]:
-- Gold Layer Star Schema Summary
SHOW TABLES IN bakehouse_retail.bakehouse_retail_gold;

In [0]:
-- Star Schema Gold Layer: Fact Table - Sales Transactions
-- Grain: One row per transaction line item
-- Includes foreign keys to dimensions + denormalized attributes for query performance
CREATE OR REPLACE TABLE bakehouse_retail.bakehouse_retail_gold.fact_sales AS
SELECT 
  t.transaction_id,
  -- Surrogate key for date dimension (format: YYYYMMDD as integer)
  CAST(DATE_FORMAT(t.transaction_date, 'yyyyMMdd') AS INT) AS date_key,
  -- Customer ID from transactions (note: different format than customers.json)
  t.customer_id,
  t.transaction_date,
  -- Foreign keys to dimension tables
  l.location_key,
  pm.payment_method_key,
  -- Denormalized product attributes (for query performance, avoids joins)
  t.product_id,
  t.product_name,
  t.category,
  -- Measures (facts)
  t.quantity,
  t.unit_price,
  t.total_amount,
  -- Denormalized attributes for common filters
  t.store_location,
  t.payment_method,
  CURRENT_TIMESTAMP() AS processed_at
FROM bakehouse_retail.bakehouse_retail_silver.transactions_clean t
-- Join to get surrogate keys from dimension tables
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_store_locations l
  ON t.store_location = l.location_name
INNER JOIN bakehouse_retail.bakehouse_retail_gold.dim_payment_methods pm
  ON t.payment_method = pm.payment_method_name;

In [0]:
-- Star Schema Gold Layer: Dimension - Date
-- Time dimension with hierarchical attributes for temporal analysis
CREATE OR REPLACE TABLE bakehouse_retail.bakehouse_retail_gold.dim_date AS
SELECT DISTINCT
  -- Primary key: date as integer YYYYMMDD format
  CAST(DATE_FORMAT(transaction_date, 'yyyyMMdd') AS INT) AS date_key,
  transaction_date AS full_date,
  -- Hierarchical time attributes for drill-down analysis
  YEAR(transaction_date) AS year,
  MONTH(transaction_date) AS month,
  DAY(transaction_date) AS day,
  QUARTER(transaction_date) AS quarter,
  WEEKOFYEAR(transaction_date) AS week_of_year,
  DAYOFWEEK(transaction_date) AS day_of_week,
  -- Human-readable attributes
  DATE_FORMAT(transaction_date, 'EEEE') AS day_name,
  DATE_FORMAT(transaction_date, 'MMMM') AS month_name,
  CURRENT_TIMESTAMP() AS processed_at
FROM bakehouse_retail.bakehouse_retail_silver.transactions_clean
ORDER BY date_key;

In [0]:
-- Star Schema Gold Layer: Dimension - Payment Methods
-- Generate surrogate keys for payment method lookup
CREATE OR REPLACE TABLE bakehouse_retail.bakehouse_retail_gold.dim_payment_methods AS
SELECT 
  ROW_NUMBER() OVER (ORDER BY payment_method) AS payment_method_key,  -- Surrogate key
  payment_method AS payment_method_name,
  CURRENT_TIMESTAMP() AS processed_at
FROM (
  SELECT DISTINCT payment_method
  FROM bakehouse_retail.bakehouse_retail_silver.transactions_clean
);

In [0]:
-- Star Schema Gold Layer: Dimension - Store Locations
-- Generate surrogate keys for store location lookup
CREATE OR REPLACE TABLE bakehouse_retail.bakehouse_retail_gold.dim_store_locations AS
SELECT 
  ROW_NUMBER() OVER (ORDER BY store_location) AS location_key,  -- Surrogate key
  store_location AS location_name,
  CURRENT_TIMESTAMP() AS processed_at
FROM (
  SELECT DISTINCT store_location
  FROM bakehouse_retail.bakehouse_retail_silver.transactions_clean
);

In [0]:
-- Star Schema Gold Layer: Dimension - Products
CREATE OR REPLACE TABLE bakehouse_retail.bakehouse_retail_gold.dim_products AS
SELECT 
  product_id,
  product_name,
  category,
  price,
  processed_at
FROM bakehouse_retail.bakehouse_retail_silver.products_clean;

In [0]:
-- Star Schema Gold Layer: Dimension - Customers
CREATE OR REPLACE TABLE bakehouse_retail.bakehouse_retail_gold.dim_customers AS
SELECT 
  customer_id,
  first_name,
  last_name,
  country,
  signup_date,
  processed_at
FROM bakehouse_retail.bakehouse_retail_silver.customers_clean;

In [0]:
SELECT DISTINCT product_name, COUNT(*) as count
FROM read_files(
  '/Volumes/bakehouse_retail/bakehouse_retail_bronze/raw/productsjson.json',
  format => 'json',
  multiLine => true
)
GROUP BY product_name
ORDER BY product_name